In [ ]:
%load_ext autoreload
%autoreload 2

import nest_asyncio
nest_asyncio.apply()

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
plt.style.use('ggplot')
params = {'legend.fontsize': 'medium',
        'figure.figsize': (18, 8),
        'axes.labelsize': 'medium',
        'axes.titlesize': 'large',
        'xtick.labelsize': 'medium',
        'ytick.labelsize': 'medium'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import datetime
import warnings
warnings.filterwarnings('ignore')

import pytz
NYC = pytz.timezone('America/New_York')

import sys
sys.path.append('../../')

# SFR Kink-Fading Backtest (Event-Driven)

Fades localized curvature anomalies in the SR3 strip using butterflies (BF), double butterflies (DF), and condors (CF).

**Signal modes:** z-score, percentile rank, cross-sectional rank, intersection  
**Regime filters:** FOMC blackout, half-life gating, ADF stationarity, vol regime  
**Structures:** 3M BF, 6M BF, 3M DF, 6M DF, 3M CF, 6M CF

---
## 1. Configuration

In [ ]:
from BT.signals.sfr_kink_fade import KinkFadeConfig, KinkStructure

CONFIG = KinkFadeConfig(
    # -- Data --
    source='BARCHART_STIRF-RL',
    curve='USD-SOFR-1D-Q12STIRT',
    n_contracts=12,
    constant_maturity=True,

    # -- Structures to trade --
    structures=[KinkStructure.BF_3M, KinkStructure.DF_3M],

    # -- Signal parameters --
    zscore_window=60,
    vol_window=20,
    carry_horizon=1,
    percentile_window=60,
    halflife_window=120,

    # -- Signal mode: "zscore", "percentile", "xsection", "intersection", "pca_residual" --
    signal_mode='zscore',
    intersection_require_all=False,

    # -- Entry filters --
    entry_min_zscore=1.5,
    entry_min_percentile=0.85,
    entry_min_xsection_rank=0.8,
    entry_max_vol=None,
    entry_require_carry=False,
    entry_min_risk_adj_roll=0.0,

    # -- Exit rules (first match wins) --
    exit_mean_reversion=True,
    exit_take_profit_zscore=None,
    exit_stop_loss_sd=2.0,
    exit_take_profit_bp=None,
    exit_stop_loss_bp=None,
    exit_max_holding_days=22,
    exit_halflife_based=False,

    # -- Portfolio --
    max_concurrent_trades=5,
    no_duplicate_structures=True,
    belly_bpv=100_000,

    # -- Regime filters --
    regime_fomc_blackout_days=5,
    regime_max_vol_percentile=None,
    regime_min_halflife_days=3.0,
    regime_max_halflife_days=120.0,
    regime_max_adf_pvalue=None,       # disabled — all p-values > 0.24 over this sample
    regime_halflife_gated=False,
    regime_vol_filter=False,
)

# Backtest date range
DATA_START = '2024-01-01'
BT_START   = '2024-06-01'
BT_END     = 'live'

structs = ', '.join(s.value for s in CONFIG.structures)
print(f'Strategy: Kink-Fade [{structs}] | Signal={CONFIG.signal_mode} | Z>{CONFIG.entry_min_zscore}')
print(f'Exit: MeanRev={CONFIG.exit_mean_reversion} | StopSD={CONFIG.exit_stop_loss_sd} | MaxHold={CONFIG.exit_max_holding_days}d')
print(f'Regime: FOMC blackout={CONFIG.regime_fomc_blackout_days}d | HL gated={CONFIG.regime_halflife_gated}')

---
## 2. Load Data & Compute Analytics

In [ ]:
from BT.signals.sfr_kink_fade import (
    run_kink_fade_analytics, compute_kink_curves,
    compute_rolling_halflife, compute_adf_snapshot,
    compute_halflife_snapshot, fit_ou_halflife,
)
from BT.signals.sfr_cal_spread_rv import SFRCalSpreadRVConfig, load_rate_panel
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from TB.TimeseriesBuilder import TimeseriesBuilder

curve_mdp = IRSwapsMDP(source=CONFIG.source)
ts_builder = TimeseriesBuilder()

sfr_config = SFRCalSpreadRVConfig(
    n_contracts=CONFIG.n_contracts,
    zscore_window=CONFIG.zscore_window,
    vol_window=CONFIG.vol_window,
    constant_maturity=CONFIG.constant_maturity,
    source=CONFIG.source,
    curve=CONFIG.curve,
)

start = NYC.localize(datetime.datetime.fromisoformat(DATA_START).replace(hour=18))
print(f'Loading rate panel ({DATA_START} -> {BT_END})...')
rates = load_rate_panel(sfr_config, start=start, end=BT_END,
                        curve_mdp=curve_mdp, ts_builder=ts_builder)
print(f'  {rates.shape[0]} dates x {rates.shape[1]} contracts')
print(f'  Columns: {list(rates.columns)}')
rates.tail(3)

In [ ]:
result = run_kink_fade_analytics(
    start=start, end=BT_END, config=CONFIG,
    rates_panel=rates, curve_mdp=curve_mdp, ts_builder=ts_builder,
    compute_halflife=True, compute_adf=True,
)

signal_table = result['signal_table']
curves = result['curves']
analytics = result['analytics']
halflife_data = result['halflife']
adf_data = result['adf']

total_signals = sum(len(v) for v in signal_table.values())
passing_signals = sum(sum(1 for s in v if s.passes_entry) for v in signal_table.values())
print(f'Signal table: {len(signal_table)} dates with signals')
print(f'Total signals: {total_signals}, passing entry: {passing_signals}')

for ks in CONFIG.structures:
    if ks in curves:
        print(f'\n{ks.value}: {len(curves[ks].columns)} structures')
        print(f'  Columns: {list(curves[ks].columns[:6])}{"..." if len(curves[ks].columns) > 6 else ""}')

---
## 3. Mean-Reversion Analytics (OU Half-Life & ADF)

In [ ]:
for ks in CONFIG.structures:
    if ks not in curves:
        continue
    curve = curves[ks]
    print(f'\n=== {ks.value.upper()} ===')

    hl_snap = compute_halflife_snapshot(curve, CONFIG.halflife_window)
    adf_snap = adf_data.get(ks, pd.Series(dtype=float))

    summary = pd.DataFrame({
        'Half-Life (d)': hl_snap.round(1),
        'ADF p-value': adf_snap.round(4),
        'Stationary?': adf_snap.apply(lambda p: 'YES' if p < 0.10 else 'no'),
        'Latest (bp)': curve.iloc[-1].round(2),
    })
    display(summary)

    # Half-life distribution
    valid_hl = hl_snap.dropna()
    if len(valid_hl) > 0:
        fig, axes = plt.subplots(1, 2, figsize=(16, 4))

        axes[0].bar(range(len(valid_hl)), valid_hl.values, color='steelblue', alpha=0.7)
        axes[0].set_xticks(range(len(valid_hl)))
        axes[0].set_xticklabels(valid_hl.index, rotation=45, ha='right', fontsize=8)
        axes[0].axhline(CONFIG.regime_min_halflife_days or 0, color='red', linestyle='--', alpha=0.5, label=f'Min HL={CONFIG.regime_min_halflife_days}d')
        axes[0].axhline(CONFIG.regime_max_halflife_days or 999, color='orange', linestyle='--', alpha=0.5, label=f'Max HL={CONFIG.regime_max_halflife_days}d')
        axes[0].set_title(f'{ks.value} -- OU Half-Life (days)')
        axes[0].set_ylabel('Half-Life (days)')
        axes[0].legend(fontsize=8)

        axes[1].bar(range(len(adf_snap)), adf_snap.values, color='coral', alpha=0.7)
        axes[1].set_xticks(range(len(adf_snap)))
        axes[1].set_xticklabels(adf_snap.index, rotation=45, ha='right', fontsize=8)
        axes[1].axhline(0.05, color='green', linestyle='--', alpha=0.5, label='5%')
        axes[1].axhline(0.10, color='orange', linestyle='--', alpha=0.5, label='10%')
        axes[1].set_title(f'{ks.value} -- ADF p-values')
        axes[1].set_ylabel('p-value')
        axes[1].legend(fontsize=8)

        plt.tight_layout()
        plt.show()

---
## 4. Run Backtest

In [ ]:
from BT.signals.sfr_kink_fade_triggers import KinkFadeEntryTrigger, KinkFadeExitTrigger
from BT.data_handler import TimeGrid
from BT.query_engine import QueryDrivenBacktest
from BT.query_strategy import QueryStrategy

entry_trigger = KinkFadeEntryTrigger(signal_table, CONFIG)
exit_trigger = KinkFadeExitTrigger(signal_table, CONFIG)

strategy = QueryStrategy(
    name='sfr_kink_fade',
    triggers=[entry_trigger, exit_trigger],
    default_mdp=curve_mdp,
)

bt_start_dt = NYC.localize(datetime.datetime.fromisoformat(BT_START).replace(hour=17))
bt_end_dt = NYC.localize(datetime.datetime.now()) if BT_END == 'live' else \
            NYC.localize(datetime.datetime.fromisoformat(BT_END).replace(hour=17))

bt_dates = pd.bdate_range(bt_start_dt, bt_end_dt, tz=NYC)
bt_datetimes = [d.to_pydatetime() for d in bt_dates]

bt = QueryDrivenBacktest(
    time_grid=TimeGrid(bt_datetimes),
    strategy=strategy,
    mdp=curve_mdp,
)

print(f'Running: {bt_start_dt.date()} to {bt_end_dt.date()} ({len(bt_datetimes)} steps)')
bt.run()
print('Done.')

---
## 5. Results

In [ ]:
mtm = pd.Series(bt.mtm_history).sort_index()
realized = bt.realized_pnl
final_mtm = mtm.iloc[-1] if len(mtm) > 0 else 0

daily_pnl = mtm.diff().dropna()
std_d = daily_pnl.std()
sharpe = (daily_pnl.mean() / std_d * np.sqrt(252)) if std_d > 0 else 0
peak = mtm.cummax()
dd = mtm - peak
max_dd = dd.min()
hit_rate = (daily_pnl > 0).mean()
n_trades = len(bt.portfolio.trades_log)
calmar = (daily_pnl.mean() * 252 / abs(max_dd)) if max_dd != 0 else 0

structs = ', '.join(s.value for s in CONFIG.structures)
print(f"{'=' * 60}")
print(f'KINK-FADING BACKTEST RESULTS [{structs}]')
print(f"{'=' * 60}")
print(f'Period:            {bt_start_dt.date()} to {bt_end_dt.date()}')
print(f'Signal mode:       {CONFIG.signal_mode}')
print(f'Total entries:     {n_trades}')
print(f'Open positions:    {len(bt.portfolio.positions)}')
print(f'Final MTM P&L:     ${final_mtm:,.0f}')
print(f'Realized P&L:      ${realized:,.0f}')
print(f'Sharpe ratio:      {sharpe:.2f}')
print(f'Calmar ratio:      {calmar:.2f}')
print(f'Max drawdown:      ${max_dd:,.0f}')
print(f'Daily hit rate:    {hit_rate:.1%}')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(18, 10), gridspec_kw={'height_ratios': [3, 1]})

ax = axes[0]
mtm.plot(ax=ax, linewidth=1.5, color='tab:cyan')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_title(f'Kink-Fade [{structs}] -- Cumulative MTM P&L ($) | Sharpe={sharpe:.2f}',
             fontweight='bold')
ax.set_ylabel('P&L ($)')
ax.grid(True, alpha=0.3)

ax = axes[1]
dd.plot(ax=ax, color='red', linewidth=1)
ax.fill_between(dd.index, dd.values, 0, color='red', alpha=0.2)
ax.set_title(f'Drawdown | Max DD = ${max_dd:,.0f}', fontweight='bold')
ax.set_ylabel('Drawdown ($)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 6. Trade Log & Exit Analysis

In [ ]:
if bt.portfolio.trades_log:
    log_df = pd.DataFrame([{
        'Date': str(getattr(o, 'timestamp', ''))[:10],
        'Action': (o.meta or {}).get('action', ''),
        'Structure': (o.meta or {}).get('structure_type', ''),
        'Tags': ', '.join((o.meta or {}).get('tags', [])[:1]),
        'Direction': (o.meta or {}).get('direction', ''),
        'Entry Z': round((o.meta or {}).get('entry_zscore', 0), 2),
        'Entry Lvl': round((o.meta or {}).get('entry_level', 0), 2),
        'Pctile': round((o.meta or {}).get('entry_percentile', 0), 2),
        'XS Rank': round((o.meta or {}).get('entry_xsection_rank', 0), 2),
    } for o in bt.portfolio.trades_log])
    print(f'Trade log ({len(log_df)} entries):')
    display(log_df.head(20))

    # Entry distribution by structure type
    print('\nEntries by structure type:')
    display(log_df.groupby('Structure').size().reset_index(name='Count'))
else:
    print('No trades executed')

In [ ]:
# Exit reason breakdown
if hasattr(bt.portfolio, 'closed_positions_log') and bt.portfolio.closed_positions_log:
    closed_df = pd.DataFrame([{
        'Opened': cp.get('opened_at', ''),
        'Closed': cp.get('closed_at', ''),
        'Hold (d)': round(cp.get('holding_period_days', 0), 1),
        'Realized': round(cp.get('realized_pnl', 0), 0),
        'Exit': (cp.get('exit_meta') or {}).get('reason', ''),
        'Structure': (cp.get('position_meta') or {}).get('structure_type', ''),
        'Direction': (cp.get('position_meta') or {}).get('direction', ''),
        'Entry Z': round((cp.get('position_meta') or {}).get('entry_zscore', 0), 2),
    } for cp in bt.portfolio.closed_positions_log])

    print(f'Closed positions: {len(closed_df)}')
    print(f'\nExit reasons:')
    display(closed_df.groupby('Exit').agg(
        Count=('Exit', 'size'),
        Avg_PnL=('Realized', 'mean'),
        Total_PnL=('Realized', 'sum'),
        Avg_Hold=('Hold (d)', 'mean'),
    ).round(0))

    print(f'\nBy direction:')
    display(closed_df.groupby('Direction').agg(
        Count=('Direction', 'size'),
        Avg_PnL=('Realized', 'mean'),
        Win_Rate=('Realized', lambda x: (x > 0).mean()),
    ).round(2))
elif hasattr(bt.portfolio, 'unwind_log') and bt.portfolio.unwind_log:
    unwind_df = pd.DataFrame([{
        'Date': str(getattr(u, 'timestamp', ''))[:10],
        'Reason': (u.meta or {}).get('reason', ''),
    } for u in bt.portfolio.unwind_log])
    print('Unwind log:')
    display(unwind_df.groupby('Reason').size().reset_index(name='Count'))
else:
    print('No closed positions logged')

---
## 7. Position Count & Signal Decay

In [ ]:
pos_counts = pd.Series({dt: len(positions) for dt, positions in bt.position_history.items()}).sort_index()

fig, ax = plt.subplots(figsize=(16, 4))
pos_counts.plot(ax=ax, kind='area', color='tab:cyan', alpha=0.4)
pos_counts.plot(ax=ax, color='tab:cyan', linewidth=1)
ax.set_title('Number of Open Positions Over Time', fontweight='bold')
ax.set_ylabel('# Positions')
ax.set_ylim(bottom=0)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Signal decay curve: avg P&L by day-after-entry
if hasattr(bt.portfolio, 'closed_positions_log') and bt.portfolio.closed_positions_log:
    hold_pnl = [(cp.get('holding_period_days', 0), cp.get('realized_pnl', 0))
                for cp in bt.portfolio.closed_positions_log]
    hold_df = pd.DataFrame(hold_pnl, columns=['hold_days', 'pnl'])
    hold_df['hold_bucket'] = pd.cut(hold_df['hold_days'], bins=[0, 3, 7, 14, 22, 44, 999],
                                     labels=['0-3d', '3-7d', '7-14d', '14-22d', '22-44d', '44d+'])

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Holding period distribution
    hold_df['hold_bucket'].value_counts().sort_index().plot(kind='bar', ax=axes[0],
        color='steelblue', alpha=0.7)
    axes[0].set_title('Holding Period Distribution')
    axes[0].set_ylabel('Count')

    # P&L by holding period
    bucket_pnl = hold_df.groupby('hold_bucket')['pnl'].mean()
    bucket_pnl.plot(kind='bar', ax=axes[1], color=['green' if v > 0 else 'red' for v in bucket_pnl])
    axes[1].set_title('Avg Realized P&L by Holding Period')
    axes[1].set_ylabel('Avg P&L ($)')
    axes[1].axhline(0, color='black', linewidth=0.5)

    plt.tight_layout()
    plt.show()

---
## 8. Structure Comparison (BF vs DF vs CF)

In [ ]:
structure_configs = {
    'BF 3M only': [KinkStructure.BF_3M],
    'DF 3M only': [KinkStructure.DF_3M],
    'BF + DF 3M': [KinkStructure.BF_3M, KinkStructure.DF_3M],
    'BF 6M only': [KinkStructure.BF_6M],
    'BF + DF + CF 3M': [KinkStructure.BF_3M, KinkStructure.DF_3M, KinkStructure.CF_3M],
}

fig, ax = plt.subplots(figsize=(18, 7))
struct_summary = []

for name, struct_list in structure_configs.items():
    cfg = KinkFadeConfig(
        source=CONFIG.source, curve=CONFIG.curve, n_contracts=CONFIG.n_contracts,
        constant_maturity=CONFIG.constant_maturity, structures=struct_list,
        zscore_window=CONFIG.zscore_window, vol_window=CONFIG.vol_window,
        signal_mode=CONFIG.signal_mode, entry_min_zscore=CONFIG.entry_min_zscore,
        exit_mean_reversion=CONFIG.exit_mean_reversion,
        exit_stop_loss_sd=CONFIG.exit_stop_loss_sd,
        exit_max_holding_days=CONFIG.exit_max_holding_days,
        max_concurrent_trades=CONFIG.max_concurrent_trades,
        regime_fomc_blackout_days=CONFIG.regime_fomc_blackout_days,
        regime_max_adf_pvalue=CONFIG.regime_max_adf_pvalue,
        belly_bpv=CONFIG.belly_bpv,
    )
    res = run_kink_fade_analytics(start=start, end=BT_END, config=cfg,
                                   rates_panel=rates, compute_halflife=False, compute_adf=False)
    st = res['signal_table']
    entry_t = KinkFadeEntryTrigger(st, cfg)
    exit_t = KinkFadeExitTrigger(st, cfg)
    strat = QueryStrategy(name=name, triggers=[entry_t, exit_t], default_mdp=curve_mdp)
    bt_run = QueryDrivenBacktest(
        time_grid=TimeGrid(bt_datetimes), strategy=strat, mdp=curve_mdp, show_progress=False,
    )
    try:
        bt_run.run()
        m = pd.Series(bt_run.mtm_history).sort_index()
        m.plot(ax=ax, label=name, linewidth=1.5)
        d = m.diff().dropna()
        s = d.std()
        struct_summary.append({
            'Strategy': name,
            'Final MTM': f'${m.iloc[-1]:,.0f}',
            'Entries': len(bt_run.portfolio.trades_log),
            'Sharpe': round(d.mean() / s * np.sqrt(252), 2) if s > 0 else 0,
            'Max DD': f'${(m - m.cummax()).min():,.0f}',
        })
    except Exception as e:
        print(f'{name}: ERROR - {e}')

ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax.set_title('Structure Comparison -- Cumulative MTM P&L ($)', fontweight='bold')
ax.set_ylabel('P&L ($)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\n=== Structure Comparison Summary ===')
display(pd.DataFrame(struct_summary).set_index('Strategy'))

---
## 9. Signal Mode Comparison

In [ ]:
signal_modes = {
    'Z-Score (z>1.5)': {'signal_mode': 'zscore', 'entry_min_zscore': 1.5},
    'Z-Score (z>2.0)': {'signal_mode': 'zscore', 'entry_min_zscore': 2.0},
    'Percentile (85th)': {'signal_mode': 'percentile', 'entry_min_percentile': 0.85},
    'XSection Rank (80th)': {'signal_mode': 'xsection', 'entry_min_xsection_rank': 0.80},
    'Intersection (AND)': {'signal_mode': 'intersection', 'intersection_require_all': True,
                           'entry_min_zscore': 1.5, 'entry_min_percentile': 0.85,
                           'entry_min_xsection_rank': 0.80},
}

fig, ax = plt.subplots(figsize=(18, 7))
mode_summary = []

for name, overrides in signal_modes.items():
    from dataclasses import replace
    cfg = replace(CONFIG, **overrides)
    res = run_kink_fade_analytics(start=start, end=BT_END, config=cfg,
                                   rates_panel=rates, compute_halflife=False, compute_adf=False)
    st = res['signal_table']
    entry_t = KinkFadeEntryTrigger(st, cfg)
    exit_t = KinkFadeExitTrigger(st, cfg)
    strat = QueryStrategy(name=name, triggers=[entry_t, exit_t], default_mdp=curve_mdp)
    bt_run = QueryDrivenBacktest(
        time_grid=TimeGrid(bt_datetimes), strategy=strat, mdp=curve_mdp, show_progress=False,
    )
    try:
        bt_run.run()
        m = pd.Series(bt_run.mtm_history).sort_index()
        m.plot(ax=ax, label=name, linewidth=1.5)
        d = m.diff().dropna()
        s = d.std()
        n_pass = sum(sum(1 for s in v if s.passes_entry) for v in st.values())
        mode_summary.append({
            'Signal Mode': name,
            'Final MTM': f'${m.iloc[-1]:,.0f}',
            'Passing Signals': n_pass,
            'Entries': len(bt_run.portfolio.trades_log),
            'Sharpe': round(d.mean() / s * np.sqrt(252), 2) if s > 0 else 0,
            'Max DD': f'${(m - m.cummax()).min():,.0f}',
        })
    except Exception as e:
        print(f'{name}: ERROR - {e}')

ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax.set_title('Signal Mode Comparison -- Cumulative MTM P&L ($)', fontweight='bold')
ax.set_ylabel('P&L ($)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\n=== Signal Mode Summary ===')
display(pd.DataFrame(mode_summary).set_index('Signal Mode'))

---
## 10. Exit Strategy Comparison

In [ ]:
exit_configs = {
    'MeanRev Only (44d)': {
        'exit_mean_reversion': True, 'exit_stop_loss_sd': None,
        'exit_take_profit_zscore': None, 'exit_max_holding_days': 44,
    },
    'MeanRev + Stop 2sd (22d)': {
        'exit_mean_reversion': True, 'exit_stop_loss_sd': 2.0,
        'exit_take_profit_zscore': None, 'exit_max_holding_days': 22,
    },
    'Fixed 10d hold': {
        'exit_mean_reversion': False, 'exit_stop_loss_sd': None,
        'exit_take_profit_zscore': None, 'exit_max_holding_days': 10,
    },
    'Fixed 21d hold': {
        'exit_mean_reversion': False, 'exit_stop_loss_sd': None,
        'exit_take_profit_zscore': None, 'exit_max_holding_days': 21,
    },
    'TP z<0.5 + Stop 2sd (22d)': {
        'exit_mean_reversion': False, 'exit_stop_loss_sd': 2.0,
        'exit_take_profit_zscore': 0.5, 'exit_max_holding_days': 22,
    },
}

fig, ax = plt.subplots(figsize=(18, 7))
exit_summary = []

# Reuse the base signal table from the primary config
base_result = run_kink_fade_analytics(start=start, end=BT_END, config=CONFIG,
                                       rates_panel=rates, compute_halflife=False, compute_adf=False)

for name, overrides in exit_configs.items():
    cfg = replace(CONFIG, **overrides)
    st = base_result['signal_table']  # same entry signals, different exits
    entry_t = KinkFadeEntryTrigger(st, cfg)
    exit_t = KinkFadeExitTrigger(st, cfg)
    strat = QueryStrategy(name=name, triggers=[entry_t, exit_t], default_mdp=curve_mdp)
    bt_run = QueryDrivenBacktest(
        time_grid=TimeGrid(bt_datetimes), strategy=strat, mdp=curve_mdp, show_progress=False,
    )
    try:
        bt_run.run()
        m = pd.Series(bt_run.mtm_history).sort_index()
        m.plot(ax=ax, label=name, linewidth=1.5)
        d = m.diff().dropna()
        s = d.std()
        exit_summary.append({
            'Exit Strategy': name,
            'Final MTM': f'${m.iloc[-1]:,.0f}',
            'Entries': len(bt_run.portfolio.trades_log),
            'Sharpe': round(d.mean() / s * np.sqrt(252), 2) if s > 0 else 0,
            'Max DD': f'${(m - m.cummax()).min():,.0f}',
        })
    except Exception as e:
        print(f'{name}: ERROR - {e}')

ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax.set_title('Exit Strategy Comparison -- Cumulative MTM P&L ($)', fontweight='bold')
ax.set_ylabel('P&L ($)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\n=== Exit Strategy Summary ===')
display(pd.DataFrame(exit_summary).set_index('Exit Strategy'))

---
## 11. Regime Filter Impact

In [ ]:
regime_configs = {
    'No regime filters': {
        'regime_fomc_blackout_days': None,
        'regime_max_adf_pvalue': None,
        'regime_halflife_gated': False,
    },
    'FOMC blackout 5d only': {
        'regime_fomc_blackout_days': 5,
        'regime_max_adf_pvalue': None,
        'regime_halflife_gated': False,
    },
    'FOMC 5d + ADF < 0.20': {
        'regime_fomc_blackout_days': 5,
        'regime_max_adf_pvalue': 0.20,
        'regime_halflife_gated': False,
    },
    'FOMC 5d + HL gated (3-120d)': {
        'regime_fomc_blackout_days': 5,
        'regime_max_adf_pvalue': None,
        'regime_halflife_gated': True,
        'regime_min_halflife_days': 3.0,
        'regime_max_halflife_days': 120.0,
    },
}

fig, ax = plt.subplots(figsize=(18, 7))
regime_summary = []

for name, overrides in regime_configs.items():
    cfg = replace(CONFIG, **overrides)
    res = run_kink_fade_analytics(start=start, end=BT_END, config=cfg,
                                   rates_panel=rates, compute_halflife=cfg.regime_halflife_gated,
                                   compute_adf=cfg.regime_max_adf_pvalue is not None)
    st = res['signal_table']
    n_pass = sum(sum(1 for s in v if s.passes_entry) for v in st.values())

    entry_t = KinkFadeEntryTrigger(st, cfg)
    exit_t = KinkFadeExitTrigger(st, cfg)
    strat = QueryStrategy(name=name, triggers=[entry_t, exit_t], default_mdp=curve_mdp)
    bt_run = QueryDrivenBacktest(
        time_grid=TimeGrid(bt_datetimes), strategy=strat, mdp=curve_mdp, show_progress=False,
    )
    try:
        bt_run.run()
        m = pd.Series(bt_run.mtm_history).sort_index()
        m.plot(ax=ax, label=name, linewidth=1.5)
        d = m.diff().dropna()
        s = d.std()
        regime_summary.append({
            'Regime Filter': name,
            'Passing Signals': n_pass,
            'Entries': len(bt_run.portfolio.trades_log),
            'Sharpe': round(d.mean() / s * np.sqrt(252), 2) if s > 0 else 0,
            'Final MTM': f'${m.iloc[-1]:,.0f}',
            'Max DD': f'${(m - m.cummax()).min():,.0f}',
        })
    except Exception as e:
        print(f'{name}: ERROR - {e}')

ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax.set_title('Regime Filter Impact -- Cumulative MTM P&L ($)', fontweight='bold')
ax.set_ylabel('P&L ($)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\n=== Regime Filter Summary ===')
display(pd.DataFrame(regime_summary).set_index('Regime Filter'))

---
## 12. Z-Score Window Sensitivity

In [ ]:
zscore_windows = [20, 40, 60, 120]

fig, ax = plt.subplots(figsize=(18, 7))
window_summary = []

for w in zscore_windows:
    name = f'ZScore window={w}d'
    cfg = replace(CONFIG, zscore_window=w)
    res = run_kink_fade_analytics(start=start, end=BT_END, config=cfg,
                                   rates_panel=rates, compute_halflife=False, compute_adf=False)
    st = res['signal_table']
    entry_t = KinkFadeEntryTrigger(st, cfg)
    exit_t = KinkFadeExitTrigger(st, cfg)
    strat = QueryStrategy(name=name, triggers=[entry_t, exit_t], default_mdp=curve_mdp)
    bt_run = QueryDrivenBacktest(
        time_grid=TimeGrid(bt_datetimes), strategy=strat, mdp=curve_mdp, show_progress=False,
    )
    try:
        bt_run.run()
        m = pd.Series(bt_run.mtm_history).sort_index()
        m.plot(ax=ax, label=name, linewidth=1.5)
        d = m.diff().dropna()
        s = d.std()
        window_summary.append({
            'Window': f'{w}d',
            'Final MTM': f'${m.iloc[-1]:,.0f}',
            'Entries': len(bt_run.portfolio.trades_log),
            'Sharpe': round(d.mean() / s * np.sqrt(252), 2) if s > 0 else 0,
            'Max DD': f'${(m - m.cummax()).min():,.0f}',
        })
    except Exception as e:
        print(f'{name}: ERROR - {e}')

ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax.set_title('Z-Score Window Sensitivity -- Cumulative MTM P&L ($)', fontweight='bold')
ax.set_ylabel('P&L ($)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\n=== Window Sensitivity Summary ===')
display(pd.DataFrame(window_summary).set_index('Window'))